In [1]:
# RCF + cellulosic ethanol without dilute-acid pretreatment

from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.cellulosic_ethanol_no_preatreatment import create_cellulosic_ethanol_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea

from biosteam import main_flowsheet as F
import biosteam as bst

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7

chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()


# ── Cellulosic ethanol — Carbohydrate_Pulp feeds directly into fermentation ─
etoh_system = create_cellulosic_ethanol_system(ins=F.Carbohydrate_Pulp)
etoh_system.simulate()

# No pretreatment_wastewater — only S401 stillage filtrate goes to WWT.
etoh_ww     = [F.unit.S401.outs[1]]
etoh_solids = [F.unit.S401.outs[0]]

# ── WWT: RCF wastewater + ethanol stillage filtrate ────────────────────────
WWT = bst.create_conventional_wastewater_treatment_system(
    'WWT',
    ins=[F.RCF_WW_OUTS] + etoh_ww,
)
for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False

# Wire WWT RO-treated water to PWC; create_all_facilities(WWT=False) leaves
# M2 (placeholder mixer) empty, so PWC would otherwise buy ~480,000 kg/hr
# of fresh water unnecessarily.
F.unit.PWC.ins[0] = WWT.outs[2]

solids_to_BT = bst.Mixer('MIX_BT_solids', ins=[WWT.outs[1]] + etoh_solids)
gas_mixer    = bst.Mixer('MIX_BT_gas',    ins=[F.RCF_PSAWASTE_OUTS, WWT.outs[0]])

BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])
BT.ins[0] = solids_to_BT.outs[0]
BT.ins[1] = gas_mixer.outs[0]


rcf_etoh_system = bst.System(
    'RCF_ETOH_system',
    path=(rcf_system, etoh_system, WWT),
    facilities=[solids_to_BT, gas_mixer, BT],
)
rcf_etoh_system.simulate()
integrated_tea = create_cellulosic_ethanol_tea(rcf_etoh_system)
F.ethanol.price = 0.76


F.cellulase.price = prices['Cellulase'] 
F.CSL.price = prices ['CSL'] 
F.DAP.price = prices['DAP'] 
F.caustic.price = prices['Caustic']
F.denaturant.price =  prices['Denaturant'] 
F.cooling_tower_chemicals.price = prices['CT_chemicals'] 


#print(f'The MSP for RCF crude oil is  {round(integrated_tea.solve_price(F.RCF_CRUDE_OUT), 3)} USD/kg')



c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.45e+03 hp), head (3.35e+03 ft), kinematic viscosity (6.08e-07 m2/s), and NPSH (4.07 ft); assuming centrigugal pump
  warn(f'{repr(

In [5]:
integrated_tea.operating_days

330.0

In [2]:
msp = round((integrated_tea.solve_price(F.RCF_CRUDE_OUT)),2)
print(f'The MSP for RCF crude is  {msp} USD/kg')

The MSP for RCF crude is  1.3 USD/kg


In [4]:
# Different sections

rcf = [F.MIX100, F.RCF_PUMP1, F.RCF_HX1, F.RCF_RXR1, F.RCF_MIX2, F.RCF_HX2, F.RCF_RXR2, F.RCF_FLSH1, F.RCF_COMP1,
F.RCF_FLSH2, F.RCF_HX3, F.RCF_PSA1, F.RCF_PUMP2, F.RCF_COL1, F.RCF_COL2, F.RCF_MIX3, F.RCF_HX4, F.RCF_FLSH3, F.RCF_MIX4, F.RCF_FLSH4]

etoh = [F.M301, F.H301, F.R301, F.DAP_storage, F.S301, F.CSL_storage, F.S302, F.R303, F.R302, F.T301, F.M304, F.D401, F.M401, F.T302, F.P401, F.H401, F.D402, F.P401, F.D403, F.H402, F.U401, F.H403, 
        F.T701, F.P701, F.T702, F.P702, F.M701, F.T703, F.P403, F.M1, F.S401 ]


other_utilities = [F.CWP, F.CT, F.FWT, F.ADP, F.PWC]

BT = [BT]

WWT = [WWT]

In [7]:
methanol_price = F.RCF_MEOH_IN.F_mass * prices['Methanol'] * integrated_tea.operating_hours
hydrogen_price = (F.RCF_H2_IN.F_mass * prices['Hydrogen']) * integrated_tea.operating_hours
                  
poplar_price = F.Poplar_In.F_mass * prices['Feedstock'] * integrated_tea.operating_hours
catalyst = (
    F.RCF_CAT_IN.F_mass * prices['NiC_catalyst']) * integrated_tea.operating_hours


cellulase_cost = F.cellulase.F_mass * prices['Cellulase'] * integrated_tea.operating_hours

# NOTE: F.denaturant has zero flow (add_denaturant=False), so it contributes $0.
# Included here for completeness — it is priced and appears in the TEA material_cost.
fermentation_chems_cost = (
    F.CSL.F_mass * prices['CSL']
    + F.DAP.F_mass * prices['DAP']
    + F.caustic.F_mass * prices['Caustic']
    + F.denaturant.F_mass * prices['Denaturant']
    + F.cooling_tower_chemicals.F_mass * prices['CT_chemicals']
) * integrated_tea.operating_hours    

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Font ──────────────────────────────────────────────────────────────────────
font_pref = ["Arial", "Liberation Sans", "DejaVu Sans"]
available = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
chosen = next((f for f in font_pref if f in available), "DejaVu Sans")
print(f"Font: {chosen}")

plt.rcParams.update({
    "font.family": chosen,
    "mathtext.fontset": "custom",
    "mathtext.rm": chosen,
    "mathtext.it": chosen,
    "mathtext.bf": chosen,
})
plt.rcParams['svg.fonttype'] = 'none'

oi_colors = [
    '#5778a4', '#e49444', '#d1615d', '#85b6b2', '#6a9f58',
    '#e7ca60', '#a87c9f', '#f1a2a9', '#967662', '#b8b0ac', "#8D86C9"
]

categories = [
    "RCF", "Oil purification", "Monomer purification", "HDO",
    "Ethanol production", "ETJ", "Boiler Turbogenerator",
    "WasteWater Treatment", "Other utilities", "Hydrogen storage"
]

values = [
    rcf_area_ic, rcf_oil_purification_ic, rcf_monomer_purification_ic,
    hdo_ic, etoh_ic, etj_ic, BT_installed_cost, WWT_installed_cost,
    other_utilities_ic, h2_storage_ic
]

# ── Figure ────────────────────────────────────────────────────────────────────
DPI      = 300
fig_w_in = 1500/1.4 / DPI
fig_h_in = 1260/1.4 / DPI

FS_TITLE  = 13
FS_TOTAL  = 13
FS_PCT    = 6
FS_LEGEND = 6

DONUT_WIDTH = 0.45   # ← controls ring thickness (0–1); increase to thicken

fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in))




DONUT_WIDTH = 0.45

def draw_donut(ax, vals, title):
    total = sum(vals)
    fracs = [v / total for v in vals]
    n = len(vals)

    wedges, _ = ax.pie(
        vals,
        colors=oi_colors[:n],
        startangle=90,
        counterclock=False,
        wedgeprops=dict(
            width=DONUT_WIDTH,
            linewidth=0,          # ← no borders between segments
            edgecolor="none",
        ),
    )

    # ── Center label ──────────────────────────────────────────────────────
    ax.text(0,  0.12, "TIC:",
            ha="center", va="center",
            fontsize=FS_TOTAL, fontweight="bold", color="#555555")
    ax.text(0, -0.16,
            f"${rcf_pure_mon_hdo_etoh_etj_system.installed_cost/1e6:.1f} MM",
            ha="center", va="center",
            fontsize=FS_TOTAL, fontweight="bold", color="#222222")

    r_outer = 1.0

    # ── Collect label info ────────────────────────────────────────────────
    label_info = []
    for i, (wedge, frac) in enumerate(zip(wedges, fracs)):
        pct   = frac * 100
        theta = np.deg2rad((wedge.theta1 + wedge.theta2) / 2)
        label_info.append({
            "theta":    theta,
            "pct":      pct,
            "cat":      categories[i],
            "is_right": np.cos(theta) >= 0,
        })

    # ── Split into left/right, sorted top → bottom by sin(theta) ─────────
    right = sorted([d for d in label_info if     d["is_right"]],
                   key=lambda d: np.sin(d["theta"]), reverse=True)
    left  = sorted([d for d in label_info if not d["is_right"]],
                   key=lambda d: np.sin(d["theta"]), reverse=True)

    # ── Evenly distribute y positions across each side ────────────────────
    Y_TOP, Y_BOT = 1.40, -1.40

    def y_positions(n_labels):
        return list(np.linspace(Y_TOP, Y_BOT, n_labels)) if n_labels > 1 else [0.0]

    right_ys = y_positions(len(right))
    left_ys  = y_positions(len(left))

    # ── Fixed x anchors ───────────────────────────────────────────────────
    # "conn" = where the diagonal meets the underline
    # "far"  = the far end of the underline
    X_R_CONN, X_R_FAR = 1.22,  2.20
    X_L_CONN, X_L_FAR = -1.22, -2.20

    # ── Draw each group ───────────────────────────────────────────────────
    GAP    = 0.05   # gap between underline and bottom of percentage text
    LINE_H = 0.22   # vertical distance between percentage and category name

    def draw_group(group, ys, is_right):
        x_conn = X_R_CONN if is_right else X_L_CONN
        x_far  = X_R_FAR  if is_right else X_L_FAR
        ha     = "left"   if is_right else "right"

        for d, y_line in zip(group, ys):
            theta = d["theta"]
            pct   = d["pct"]
            cat   = d["cat"]

            # Point just outside the outer ring
            x0 = (r_outer + 0.02) * np.cos(theta)
            y0 = (r_outer + 0.02) * np.sin(theta)

            # ── Diagonal leader line ──────────────────────────────────────
            ax.plot([x0, x_conn], [y0, y_line],
                    color="#333333", lw=0.9, solid_capstyle="round",
                    clip_on=False, zorder=5)

            # ── Horizontal underline ──────────────────────────────────────
            ax.plot([x_conn, x_far], [y_line, y_line],
                    color="#333333", lw=0.9, solid_capstyle="round",
                    clip_on=False, zorder=5)

            # ── Percentage (just above underline) ─────────────────────────
            ax.text(x_conn, y_line + GAP,
                    f"{pct:.1f}%",
                    ha=ha, va="bottom",
                    fontsize=FS_PCT,
                    color="#555555",
                    clip_on=False)

            # ── Category name (bold, above percentage) ────────────────────
            ax.text(x_conn, y_line + GAP + LINE_H,
                    cat,
                    ha=ha, va="bottom",
                    fontsize=FS_PCT,
                    fontweight="bold",
                    color="#222222",
                    clip_on=False)

    draw_group(right, right_ys, is_right=True)
    draw_group(left,  left_ys,  is_right=False)

    ax.set_title(title, fontsize=FS_TITLE, fontweight="bold", pad=10)
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-1.9, 1.75)


draw_donut(ax, values, "Installed Cost Breakdown")

# ── DELETE the fig.legend(...) block entirely ──────────────────────────────

plt.rcParams['svg.fonttype'] = 'none'
fig.tight_layout(rect=[0, 0.02, 1, 1])   # was [0, 0.14, 1, 1]
fig.savefig("installed_cost_breakdown_5.svg", format="svg", bbox_inches="tight")

In [3]:
(0.76*753)/264.172

2.1663158851051585